In [287]:
import pandas as pd
import numpy as np
import os


In [288]:
stock = pd.read_csv("d:\Downloads\stock2_prices.csv")
macro = pd.read_csv("d:\Downloads\macro2_prices.csv")

In [289]:
stock = stock[stock.iloc[:,0] != 'Date']
macro = macro[macro.iloc[:,0] != 'Date']

In [290]:
stock.columns = [
    'Date',

    'AAPL_Close',
    'AAPL_AdjClose',
    'AAPL_Volume',

    'AMZN_Close',
    'AMZN_AdjClose',
    'AMZN_Volume',

    'JNJ_Close',
    'JNJ_AdjClose',
    'JNJ_Volume',

    'JPM_Close',
    'JPM_AdjClose',
    'JPM_Volume',

    'META_Close',
    'META_AdjClose',
    'META_Volume',

    'MSFT_Close',
    'MSFT_AdjClose',
    'MSFT_Volume',

    'NVDA_Close',
    'NVDA_AdjClose',
    'NVDA_Volume',

    'TSLA_Close',
    'TSLA_AdjClose',
    'TSLA_Volume'
]


In [291]:
macro.columns = [
    'Date',

    'SP500_Close',
    'SP500_AdjClose',
    'SP500_Volume',

    'Gold_Close',
    'Gold_AdjClose',
    'Gold_Volume',

    'Oil_Close',
    'Oil_AdjClose',
    'Oil_Volume'
]


In [292]:
macro_cols = [
    "SP500_Close",
    "SP500_AdjClose",
    "SP500_Volume",
    "Gold_Close",
    "Gold_AdjClose",
    "Gold_Volume",
    "Oil_Close",
    "Oil_AdjClose",
    "Oil_Volume"
]


In [293]:
data = pd.read_excel("d:\Documents\IBFM\cleaned_data_updated.xlsx")


In [294]:
stock['Date'] = pd.to_datetime(
    stock['Date'],
    errors='coerce'
)

macro['Date'] = pd.to_datetime(
    macro['Date'],
    errors='coerce'
)

stock.dropna(subset=['Date'], inplace=True)
macro.dropna(subset=['Date'], inplace=True)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8256\2578468495.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  stock['Date'] = pd.to_datetime(
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8256\2578468495.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  macro['Date'] = pd.to_datetime(


In [295]:
print(data[[
    "SP500_return",
    "Gold_return",
    "Oil_return"
]].head())


   SP500_return  Gold_return  Oil_return
0           NaN          NaN         NaN
1     -0.005945    -0.010680    0.002773
2     -0.001272     0.004253   -0.013789
3     -0.027392    -0.000869   -0.002767
4     -0.002358    -0.004021   -0.020041


In [296]:
df = data

In [297]:
cols_to_drop = [col for col in data.columns 
                if '_Close_return' in col 
                or '_AdjClose_return' in col
                or '_Volume_return' in col]

data = data.drop(columns=cols_to_drop)

In [298]:
data['SP500_return'] = data['SP500_Close'].pct_change()
data['Gold_return'] = data['Gold_Close'].pct_change()
data['Oil_return'] = data['Oil_Close'].pct_change()

In [299]:
data['SP500_MA7'] = data['SP500_Close'].rolling(7).mean()
data['SP500_MA30'] = data['SP500_Close'].rolling(30).mean()

data['Gold_MA7'] = data['Gold_Close'].rolling(7).mean()
data['Gold_MA30'] = data['Gold_Close'].rolling(30).mean()

data['Oil_MA7'] = data['Oil_Close'].rolling(7).mean()
data['Oil_MA30'] = data['Oil_Close'].rolling(30).mean()

In [300]:
for asset in ['SP500', 'Gold', 'Oil']:
    data[f'{asset}_return'] = data[f'{asset}_Close'].pct_change()
    data[f'{asset}_MA7'] = data[f'{asset}_Close'].rolling(7).mean()
    data[f'{asset}_MA30'] = data[f'{asset}_Close'].rolling(30).mean()
    data[f'{asset}_volatility'] = data[f'{asset}_return'].rolling(30).std()

In [301]:
data = data.dropna()

In [302]:
data = data.reset_index(drop=True)

In [303]:
data.to_excel("cleaned_data_updated.xlsx", index=False)

print("DONE")

DONE


In [304]:
for col in stock.columns[1:]:
    stock[col] = pd.to_numeric(
        stock[col],
        errors='coerce'
    )

for col in macro.columns[1:]:
    macro[col] = pd.to_numeric(
        macro[col],
        errors='coerce'
    )


In [305]:
data = pd.merge(
    stock,
    macro,
    on='Date',
    how='left'
)

In [306]:
data.set_index('Date', inplace=True)


In [307]:
data.fillna(method='ffill', inplace=True)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8256\1984096990.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data.fillna(method='ffill', inplace=True)


In [308]:
data.drop_duplicates(inplace=True)

In [309]:

close_cols = [
    'AAPL_Close',
    'AMZN_Close',
    'JNJ_Close',
    'JPM_Close',
    'META_Close',
    'MSFT_Close',
    'NVDA_Close',
    'TSLA_Close'
]

In [310]:
for col in close_cols:

    stock_name = col.replace('_Close', '')

    # RETURN
    data[f'{stock_name}_return'] = (
        data[col]
        .pct_change()
    )

    # MA7
    data[f'{stock_name}_MA7'] = (
        data[col]
        .rolling(window=7)
        .mean()
    )

    # MA30
    data[f'{stock_name}_MA30'] = (
        data[col]
        .rolling(window=30)
        .mean()
    )

    # VOLATILITY
    data[f'{stock_name}_volatility'] = (
        data[f'{stock_name}_return']
        .rolling(window=7)
        .std()
    )


In [311]:
print(data.columns.tolist())

['AAPL_Close', 'AAPL_AdjClose', 'AAPL_Volume', 'AMZN_Close', 'AMZN_AdjClose', 'AMZN_Volume', 'JNJ_Close', 'JNJ_AdjClose', 'JNJ_Volume', 'JPM_Close', 'JPM_AdjClose', 'JPM_Volume', 'META_Close', 'META_AdjClose', 'META_Volume', 'MSFT_Close', 'MSFT_AdjClose', 'MSFT_Volume', 'NVDA_Close', 'NVDA_AdjClose', 'NVDA_Volume', 'TSLA_Close', 'TSLA_AdjClose', 'TSLA_Volume', 'SP500_Close', 'SP500_AdjClose', 'SP500_Volume', 'Gold_Close', 'Gold_AdjClose', 'Gold_Volume', 'Oil_Close', 'Oil_AdjClose', 'Oil_Volume', 'AAPL_return', 'AAPL_MA7', 'AAPL_MA30', 'AAPL_volatility', 'AMZN_return', 'AMZN_MA7', 'AMZN_MA30', 'AMZN_volatility', 'JNJ_return', 'JNJ_MA7', 'JNJ_MA30', 'JNJ_volatility', 'JPM_return', 'JPM_MA7', 'JPM_MA30', 'JPM_volatility', 'META_return', 'META_MA7', 'META_MA30', 'META_volatility', 'MSFT_return', 'MSFT_MA7', 'MSFT_MA30', 'MSFT_volatility', 'NVDA_return', 'NVDA_MA7', 'NVDA_MA30', 'NVDA_volatility', 'TSLA_return', 'TSLA_MA7', 'TSLA_MA30', 'TSLA_volatility']


In [312]:
data.dropna(inplace=True)

In [313]:
data.reset_index(inplace=True)

In [314]:
output_folder = r"d:\Documents\IBFM"

output_file = os.path.join(
    output_folder,
    "cleaned_data.csv"
)

data.to_csv(output_file, index=False)


In [315]:
print("===================================")
print("DATA CLEANING COMPLETED")
print("===================================")

print("\nDataset Shape:")
print(data.shape)

print("\nFirst 5 Rows:")
print(data.head())

print("\nSaved file:")
print(output_file)

DATA CLEANING COMPLETED

Dataset Shape:
(473, 66)

First 5 Rows:
        Date  AAPL_Close  AAPL_AdjClose  AAPL_Volume  AMZN_Close  \
0 2023-02-14  150.873291     153.199997     61707600   99.699997   
1 2023-02-15  152.970901     155.330002     65573800  101.160004   
2 2023-02-16  151.375504     153.710007     68167900   98.150002   
3 2023-02-17  150.233139     152.550003     59144100   97.199997   
4 2023-02-21  146.224945     148.479996     58867200   94.580002   

   AMZN_AdjClose  AMZN_Volume   JNJ_Close  JNJ_AdjClose  JNJ_Volume  ...  \
0      99.699997     56202900  147.131592    162.039993     6316000  ...   
1     101.160004     47957600  144.707275    159.369995    12621800  ...   
2      98.150002     56339200  143.681229    158.240005    11177000  ...   
3      97.199997     60029400  146.680908    160.389999    12401900  ...   
4      94.580002     56580400  144.495178    158.000000     9423900  ...   

    MSFT_MA30  MSFT_volatility  NVDA_return   NVDA_MA7  NVDA_MA30  \


In [316]:
output_file = os.path.join(output_folder, "cleaned_data.xlsx")
data.to_excel(output_file, index=False)

In [317]:
print(data.columns)

Index(['Date', 'AAPL_Close', 'AAPL_AdjClose', 'AAPL_Volume', 'AMZN_Close',
       'AMZN_AdjClose', 'AMZN_Volume', 'JNJ_Close', 'JNJ_AdjClose',
       'JNJ_Volume', 'JPM_Close', 'JPM_AdjClose', 'JPM_Volume', 'META_Close',
       'META_AdjClose', 'META_Volume', 'MSFT_Close', 'MSFT_AdjClose',
       'MSFT_Volume', 'NVDA_Close', 'NVDA_AdjClose', 'NVDA_Volume',
       'TSLA_Close', 'TSLA_AdjClose', 'TSLA_Volume', 'SP500_Close',
       'SP500_AdjClose', 'SP500_Volume', 'Gold_Close', 'Gold_AdjClose',
       'Gold_Volume', 'Oil_Close', 'Oil_AdjClose', 'Oil_Volume', 'AAPL_return',
       'AAPL_MA7', 'AAPL_MA30', 'AAPL_volatility', 'AMZN_return', 'AMZN_MA7',
       'AMZN_MA30', 'AMZN_volatility', 'JNJ_return', 'JNJ_MA7', 'JNJ_MA30',
       'JNJ_volatility', 'JPM_return', 'JPM_MA7', 'JPM_MA30', 'JPM_volatility',
       'META_return', 'META_MA7', 'META_MA30', 'META_volatility',
       'MSFT_return', 'MSFT_MA7', 'MSFT_MA30', 'MSFT_volatility',
       'NVDA_return', 'NVDA_MA7', 'NVDA_MA30', 'NVDA_vo